In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\Usuario\OneDrive\Documentos\GitHub\ENTREGABLES-BIT-Johann_Moreno\Data_Bases\animal_data_dirty1.csv", sep=';')




In [18]:
#• Cuántas filas y columnas tiene el DataFrame?
print(f"El DataFrame tiene {df.shape[0]} filas y {df.shape[1]} columnas.")

El DataFrame tiene 1011 filas y 11 columnas.


In [17]:
#• Cuáles son las columnas y qué tipo de dato tiene cada una?
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Animal type       991 non-null    str    
 1   Country           999 non-null    str    
 2   Weight kg         984 non-null    float64
 3   Body Length cm    984 non-null    float64
 4   Gender            992 non-null    str    
 5   Animal code       0 non-null      float64
 6   Latitude          913 non-null    float64
 7   Longitude         913 non-null    float64
 8   Animal name       52 non-null     str    
 9   Observation date  1011 non-null   str    
 10  Data compiled by  1011 non-null   str    
dtypes: float64(5), str(6)
memory usage: 87.0 KB


In [ ]:
#• Cuántos valores faltantes existen por columna?
df.isnull().sum()

Animal type           20
Country               12
Weight kg             27
Body Length cm        27
Gender                19
Animal code         1011
Latitude              98
Longitude             98
Animal name          959
Observation date       0
Data compiled by       0
dtype: int64

In [20]:
#• Cuántos registros duplicados existen?
df.duplicated().sum()

np.int64(167)

In [27]:
#• Utiliza unique() o value_counts() para identificar categorías inconsistentes o valores sospechosos.
for columna in df.columns:
    print(f"--- Valores únicos en: {columna} ---")
    print(df[columna].unique())
    print("\n" + "="*40 + "\n")

--- Valores únicos en: Animal type ---
<StringArray>
[              nan,  'European bison', 'European bison™', 'European bisson',
 'European buster',            'lynx',           'lynx?',     'red squirel',
    'red squirrel',   'red squirrell',        'hedgehog',        'wedgehod',
        'ledgehod']
Length: 13, dtype: str


--- Valores únicos en: Country ---
<StringArray>
[             nan,         'Poland',        'Hungary',             'PL',
        'Germany',       'Slovakia', 'Czech Republic',          'Czech',
         'Hungry',             'HU',        'Austria',      'Australia',
             'CZ',             'DE',             'CC']
Length: 15, dtype: str


--- Valores únicos en: Weight kg ---
[      nan  9.30e+02  9.09e+02  5.81e+02  9.00e+02  6.20e+02  2.40e+01
  2.30e+01  3.08e-01  3.48e-01  3.16e-01  3.40e-01  3.33e-01  2.65e-01
  2.61e-01  3.17e-01  3.28e-01  3.05e-01  2.94e-01  2.50e-01  2.62e-01
  2.67e-01  2.69e-01  2.70e-01  3.20e-01  2.56e-01  2.45e-01  2.92e-01
  

In [ ]:
#Tres problemas de calidad que encontré en la base de datos:

#Hay una columna que no sirve para nada: La columna de Animal code está totalmente vacía (tiene 0 datos registrados), por lo que está ocupando espacio en vano y hay que descartarla.

#Errores de escritura en los nombres de los animales: En la columna Animal type hay palabras mal escritas, espacios de más o símbolos raros que hacen que el sistema piense que son animales diferentes cuando en realidad son el mismo. Por ejemplo, aparece red squirrel escrito de varias formas distintas, o errores de dedo claros como ledgehod en lugar de hedgehog.   

#Muchísima información incompleta (datos faltantes): Faltan muchos datos a lo largo de toda la tabla. El caso más grave es Animal name, donde prácticamente no hay registros (faltan 959 nombres), y también faltan bastantes datos en las coordenadas de ubicación (Latitude y Longitude).

In [28]:
# 1. Eliminar la columna 'Animal code' porque está 100% vacía
df = df.drop(columns=['Animal code'])

# 2. Rellenar los nombres faltantes en 'Animal name' con "Desconocido"
df['Animal name'] = df['Animal name'].fillna('Desconocido')

# 3. Para las demás columnas con nulos (peso, tamaño, ubicación, género, etc.), 
# podemos rellenar los valores numéricos con la mediana.
df['Weight kg'] = df['Weight kg'].fillna(df['Weight kg'].median())
df['Body Length cm'] = df['Body Length cm'].fillna(df['Body Length cm'].median())

# Verificamos que ya no queden nulos críticos
print(df.isnull().sum())

Animal type         20
Country             12
Weight kg            0
Body Length cm       0
Gender              19
Latitude            98
Longitude           98
Animal name          0
Observation date     0
Data compiled by     0
dtype: int64


In [29]:
# Ver cuántos duplicados hay antes de eliminarlos
print("Duplicados encontrados:", df.duplicated().sum())

# Eliminar los registros duplicados
df = df.drop_duplicates()

# Confirmar que el total de duplicados ahora es 0
print("Duplicados después de limpiar:", df.duplicated().sum())

Duplicados encontrados: 167
Duplicados después de limpiar: 0


In [ ]:
# Convertir 'Observation date' a formato datetime
df['Observation date'] = pd.to_datetime(df['Observation date'], errors='coerce',dayfirst=True)

# Verificamos el nuevo tipo de dato con .dtypes()
print(df[['Observation date']].dtypes)



Observation date    datetime64[us]
dtype: object


In [35]:
# Reemplazar los valores mal escritos o con símbolos por el nombre correcto y limpio
df['Animal type'] = df['Animal type'].replace({
    'red squirrell': 'red squirrel',
    'red squirle': 'red squirrel',
    'lynx?': 'lynx',
    'European bison™': 'European bison',
    'European bisson': 'European bison',
    'European buster': 'European bison',
    'ledgehod': 'hedgehog',
    'wedgehod': 'hedgehog'
})

# Comprobamos que las categorías ya quedaron limpias y unificadas
print(df['Animal type'].value_counts())

Animal type
red squirrel      403
hedgehog          278
lynx               71
European bison     63
red squirel        10
Name: count, dtype: int64


In [36]:
print("--- DIMENSIONES FINALES DEL DATAFRAME ---")
print(df.shape)

print("\n--- VALORES NULOS RESTANTES ---")
print(df.isnull().sum())

print("\n--- TIPOS DE DATOS ACTUALIZADOS ---")
print(df.dtypes)

--- DIMENSIONES FINALES DEL DATAFRAME ---
(844, 10)

--- VALORES NULOS RESTANTES ---
Animal type         19
Country             11
Weight kg            0
Body Length cm       0
Gender              15
Latitude            92
Longitude           92
Animal name          0
Observation date    35
Data compiled by     0
dtype: int64

--- TIPOS DE DATOS ACTUALIZADOS ---
Animal type                    str
Country                        str
Weight kg                  float64
Body Length cm             float64
Gender                         str
Latitude                   float64
Longitude                  float64
Animal name                    str
Observation date    datetime64[us]
Data compiled by               str
dtype: object


In [38]:
# Rellenar los nulos restantes que se ven en la consola
df['Animal type'] = df['Animal type'].fillna('Desconocido')
df['Country'] = df['Country'].fillna('Desconocido')
df['Gender'] = df['Gender'].fillna('No especificado')

# Para las coordenadas y fechas que faltan, podemos rellenar o dejar una fecha por defecto
df['Observation date'] = df['Observation date'].fillna(df['Observation date'].mode()[0]) # Rellenar con la fecha más frecuente
df['Latitude'] = df['Latitude'].fillna(df['Latitude'].median())
df['Longitude'] = df['Longitude'].fillna(df['Longitude'].median())

# Vuelve a comprobar los nulos para confirmar que todo está en 0
print(df.isnull().sum())

Animal type         0
Country             0
Weight kg           0
Body Length cm      0
Gender              0
Latitude            0
Longitude           0
Animal name         0
Observation date    0
Data compiled by    0
dtype: int64


In [39]:

# 1. Crear una variable utilizando np.where()
# Contexto de animales: Clasificaremos el peso en "Pesado" o "Ligero" 
# usando como umbral el peso promedio (por ejemplo, 50 kg, o calculándolo con numpy).
peso_promedio = np.nanmean(df["Weight kg"])

# np.where(condicion, valor_si_es_verdadero, valor_si_es_falso)
df['Categoria_Peso'] = np.where(df['Weight kg'] > peso_promedio, 'Pesado', 'Ligero')

# 2. Utilizar una función estadística de NumPy (ej. np.nanmean) sobre una variable numérica
# Calculamos la mediana del tamaño corporal (Body Length cm) ignorando los nulos si los hubiera
longitud_mediana = np.nanmedian(df["Body Length cm"])

print(f"--- RESULTADOS DE NUMPY ---")
print(f"El peso promedio de referencia fue: {peso_promedio:.2f} kg")
print(f"La mediana del largo corporal (Body Length cm) es: {longitud_mediana} cm")

# Mostramos las primeras filas para ver la nueva columna creada con np.where
display(df[['Animal type', 'Weight kg', 'Categoria_Peso', 'Body Length cm']].head())

--- RESULTADOS DE NUMPY ---
El peso promedio de referencia fue: 46.29 kg
La mediana del largo corporal (Body Length cm) es: 21.0 cm


,Animal type,Weight kg,Categoria_Peso,Body Length cm
0,Desconocido,0.3315,Ligero,21.0
1,Desconocido,0.3315,Ligero,21.0
2,European bison,930.0000,Pesado,335.0
3,European bison,909.0000,Pesado,311.0
4,European bison,581.0000,Pesado,277.0
